In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!ls /content/drive/MyDrive

Mounted at /content/drive
 20191216_122821.mp4		  logic
 20200101_074036.mp4		  math
 20200324_211649.mp4		 'Screenshot_20220307-174728_().jpg'
 Classroom			  대마도여행250224.gmap
'Colab Notebooks'		  여수여행_241216.gmap
'Gantt Chart_07의 사본.gslides'  '역사 보고서.show'
 gptneo-1.3B-finetuned		  오키나와여행_241230.gmap
 gptneo-1.3B-masked		  통지서.html
 gptneo-finetuned-qa		  홍콩여행_240121.gmap
 knowledge


In [ ]:
# 1. 필요한 패키지 설치
!pip install --upgrade transformers datasets accelerate --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 123.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import GPT2LMHeadModel, PreTrainedTokenizerFast, Trainer, TrainingArguments, DataCollatorForLanguageModeling

# 모델 및 토크나이저 로딩
model_name = "skt/kogpt2-base-v2"
tokenizer = PreTrainedTokenizerFast.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

tokenizer.add_special_tokens({'eos_token': '</s>', 'pad_token': '<pad>'})
model.resize_token_embeddings(len(tokenizer))

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json:   0%|          | 0.00/2.83M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPT2Tokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


pytorch_model.bin:   0%|          | 0.00/513M [00:00<?, ?B/s]

Embedding(51200, 768)

In [ ]:
# 3. 수학문제 데이터
from google.colab import files
import json

uploaded = files.upload()

with open("augmented_math.json", "r", encoding="utf-8") as f:
    math_data = json.load(f)

Saving augmented_math.json to augmented_math.json


In [ ]:
# 3. 상식문제 데이터
from google.colab import files
import json

uploaded = files.upload()

with open("augmented_knowledge.json", "r", encoding="utf-8") as f:
    knowledge_data = json.load(f)

Saving augmented_knowledge.json to augmented_knowledge.json


In [ ]:
# 3. 논리문제 데이터
from google.colab import files
import json

uploaded = files.upload()

with open("augmented_logic.json", "r", encoding="utf-8") as f:
    logic_data = json.load(f)

model.safetensors:   0%|          | 0.00/513M [00:00<?, ?B/s]

Saving augmented_logic.json to augmented_logic.json


In [ ]:
# 3. 정답이 없는 문제 데이터
from google.colab import files
import json

uploaded = files.upload()

with open("augmented_open_problem.json", "r", encoding="utf-8") as f:
    open_problem_data = json.load(f)

Saving augmented_open_problem.json to augmented_open_problem.json


In [ ]:
# 4. Math 문제 텍스트 생성
formatted_data = []
for item in math_data:
    prompt = f"문제: {item['Question']}\n"
    formatted_data.append({"Text": prompt})

In [ ]:
# 4. Knowledge 문제 텍스트 생성
formatted_data = []
for item in knowledge_data:
    prompt = f"문제: {item['Question']}\n"
    formatted_data.append({"Text": prompt})

In [ ]:
# 4. Logic 문제 텍스트 생성
formatted_data = []
for item in logic_data:
    prompt = f"문제: {item['Question']}\n"
    formatted_data.append({"Text": prompt})

In [ ]:
# 4. Open Problem 문제 텍스트 생성
formatted_data = []
for item in open_problem_data:
    prompt = f"문제: {item['Question']}\n"
    formatted_data.append({"Text": prompt})

In [ ]:
# 4. Math 문제+답 텍스트 생성
formatted_data = []
for item in math_data:
    prompt = f"문제: {item['Question']}\n정답: {item['Answer']}"
    formatted_data.append({"Text": prompt})

In [ ]:
# 토크나이즈 함수
def tokenize(example):
    return tokenizer(example["Text"], padding="max_length", truncation=True, max_length=512)

In [ ]:
# 6. Dataset 변환 및 전처리
from datasets import Dataset

dataset = Dataset.from_list(formatted_data)
tokenized_dataset = dataset.map(tokenize, batched=False)

Map:   0%|          | 0/23 [00:00<?, ? examples/s]

In [ ]:
import transformers
import inspect

print(transformers.__file__)  # 실제 불러온 transformers 위치 확인
print(inspect.getfile(transformers.TrainingArguments))  # TrainingArguments가 로드된 경로

/usr/local/lib/python3.11/dist-packages/transformers/__init__.py
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py


In [ ]:
# 7. Trainer 준비
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./kogpt2-finetuned",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=1,
    logging_steps=10,
    eval_strategy="no",
    fp16=torch.cuda.is_available(),
    overwrite_output_dir=True
)

In [ ]:
# 데이터 collator
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
# 8. Trainer 구성 & 학습 실행 (Math)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

<ipython-input-12-ceccad0ce5d7>:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: circlehalf17 (circlehalf17-no-job) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.972300
20,3.454500
30,3.456100
40,2.911300
50,3.295600
60,2.881400
70,2.569300
80,2.327300
90,1.834200
100,1.463700


TrainOutput(global_step=850, training_loss=0.875049848276026, metrics={'train_runtime': 156.6536, 'train_samples_per_second': 10.852, 'train_steps_per_second': 5.426, 'total_flos': 444196454400000.0, 'train_loss': 0.875049848276026, 'epoch': 10.0})

In [ ]:
# 8. Trainer 구성 & 학습 실행 (Knowledge)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

<ipython-input-11-5294ec0e91f6>:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: circlehalf17 (circlehalf17-no-job) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,4.690100
20,4.110400
30,3.678300
40,3.270400
50,3.349300
60,3.260600
70,3.067100
80,3.295500
90,2.656500
100,3.028600


TrainOutput(global_step=1060, training_loss=1.0194026485928949, metrics={'train_runtime': 232.1961, 'train_samples_per_second': 9.13, 'train_steps_per_second': 4.565, 'total_flos': 553939107840000.0, 'train_loss': 1.0194026485928949, 'epoch': 10.0})

In [ ]:
# 8. Trainer 구성 & 학습 실행 (Logic)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

<ipython-input-12-08f308f2cd0f>:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: circlehalf17 (circlehalf17-no-job) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,4.556400
20,4.139200
30,3.732500
40,3.415900
50,3.265800
60,3.145500
70,2.995000
80,3.020000
90,1.990100
100,1.795400


TrainOutput(global_step=820, training_loss=0.9301771078167892, metrics={'train_runtime': 153.8174, 'train_samples_per_second': 10.597, 'train_steps_per_second': 5.331, 'total_flos': 425906012160000.0, 'train_loss': 0.9301771078167892, 'epoch': 10.0})

In [ ]:
# 8. Trainer 구성 & 학습 실행 (Open Problem)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

<ipython-input-13-cd8c79ce932f>:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: circlehalf17 (circlehalf17-no-job) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.233600
20,1.047900
30,1.069300
40,0.604000
50,0.515100
60,0.412500
70,0.369500
80,0.364500
90,0.315400
100,0.278800


TrainOutput(global_step=120, training_loss=0.7245022177696228, metrics={'train_runtime': 37.493, 'train_samples_per_second': 6.134, 'train_steps_per_second': 3.201, 'total_flos': 60097167360000.0, 'train_loss': 0.7245022177696228, 'epoch': 10.0})

In [ ]:
model.save_pretrained("/content/drive/MyDrive/math")
tokenizer.save_pretrained("/content/drive/MyDrive/math")

('/content/drive/MyDrive/math/tokenizer_config.json',
 '/content/drive/MyDrive/math/special_tokens_map.json',
 '/content/drive/MyDrive/math/tokenizer.json')

In [ ]:
model.save_pretrained("/content/drive/MyDrive/knowledge")
tokenizer.save_pretrained("/content/drive/MyDrive/knowledge")

('/content/drive/MyDrive/knowledge/tokenizer_config.json',
 '/content/drive/MyDrive/knowledge/special_tokens_map.json',
 '/content/drive/MyDrive/knowledge/tokenizer.json')

In [ ]:
model.save_pretrained("/content/drive/MyDrive/logic")
tokenizer.save_pretrained("/content/drive/MyDrive/logic")

('/content/drive/MyDrive/logic/tokenizer_config.json',
 '/content/drive/MyDrive/logic/special_tokens_map.json',
 '/content/drive/MyDrive/logic/tokenizer.json')

In [ ]:
model.save_pretrained("/content/drive/MyDrive/open_problem")
tokenizer.save_pretrained("/content/drive/MyDrive/open_problem")

('/content/drive/MyDrive/open_problem/tokenizer_config.json',
 '/content/drive/MyDrive/open_problem/special_tokens_map.json',
 '/content/drive/MyDrive/open_problem/tokenizer.json')